# Quant firm education backgrounds (LinkedIn)

Scrape LinkedIn company employee lists, pull each person's **education**, and summarize which schools appear most often.

**Why you saw “No profile data could be extracted”:** LinkedIn blocks unauthenticated scraping. You need either (1) a **logged-in browser** (Selenium cells below), or (2) **`PROXYCURL_API_KEY`** in your environment.

```bash
pip install linkedin-scraper selenium webdriver-manager pandas matplotlib requests
```

In [ ]:
import os
import re
import time
from collections import Counter
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests

COMPANIES = {
    "Jane Street": "https://www.linkedin.com/company/jane-street/",
    "Hudson River Trading": "https://www.linkedin.com/company/hudson-river-trading/",
}

MAX_EMPLOYEES_PER_COMPANY = 30
PAUSE_BETWEEN_PROFILES_SEC = 2.0
OUTPUT_DIR = Path("schools_output")
OUTPUT_DIR.mkdir(exist_ok=True)

PROXYCURL_API_KEY = os.environ.get("PROXYCURL_API_KEY", "").strip()

## 1. Log in (Selenium)

Run the next two cells: a Chrome window opens → **log in to LinkedIn** → run the auth check cell.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

try:
    from webdriver_manager.chrome import ChromeDriverManager
    service = Service(ChromeDriverManager().install())
except Exception:
    service = Service()

opts = Options()
opts.add_argument("--start-maximized")

driver = webdriver.Chrome(service=service, options=opts)
driver.get("https://www.linkedin.com/login")
print("Log in in the browser, then run the auth check cell.")

In [ ]:
driver.get("https://www.linkedin.com/feed/")
time.sleep(2)
url = driver.current_url
logged_in = "login" not in url and "authwall" not in url
print("URL:", url)
print("Logged in:", logged_in)
if not logged_in:
    raise RuntimeError("Not logged in — finish login in the browser and re-run this cell.")

## 2. Extract employees + education

In [ ]:
def _slug_from_company_url(company_url: str) -> str:
    return company_url.rstrip("/").split("/company/")[-1].split("?")[0]


def fetch_employees_selenium(company_name: str, company_url: str, driver, limit: int):
    """Uses linkedin_scraper with an existing logged-in Selenium driver."""
    from linkedin_scraper import CompanyScraper

    scraper = CompanyScraper(driver)
    print(f"Extracting profiles for: {company_name}...")
    company = scraper.scrape(company_url)
    employees = company.employees or []
    if not employees:
        # Fallback: open /people/ page (often more reliable than overview scrape)
        slug = _slug_from_company_url(company_url)
        people_url = f"https://www.linkedin.com/company/{slug}/people/"
        driver.get(people_url)
        time.sleep(3)
        links = driver.find_elements("css selector", "a[href*='/in/']")
        seen = set()
        employees = []
        for el in links:
            href = el.get_attribute("href") or ""
            if "/in/" not in href or href in seen:
                continue
            seen.add(href)
            name = (el.text or "").strip().split("\n")[0]
            if name:
                employees.append({"name": name, "linkedin_url": href.split("?")[0]})
            if len(employees) >= limit:
                break
    rows = []
    for emp in employees[:limit]:
        url = getattr(emp, "linkedin_url", None) or emp.get("linkedin_url")
        name = getattr(emp, "name", None) or emp.get("name")
        if not url:
            continue
        from linkedin_scraper import PersonScraper

        person_scraper = PersonScraper(driver)
        try:
            person = person_scraper.scrape(url)
        except Exception as e:
            print(f"  skip {name}: {e}")
            continue
        for edu in person.educations or []:
            school = (getattr(edu, "institution_name", None) or getattr(edu, "school", None) or "").strip()
            if school:
                rows.append({
                    "company": company_name,
                    "name": name,
                    "profile_url": url,
                    "school": school,
                    "degree": getattr(edu, "degree", None),
                })
        time.sleep(PAUSE_BETWEEN_PROFILES_SEC)
    return rows


def fetch_employees_proxycurl(company_name: str, company_url: str, limit: int):
    """Paid API — no browser login. Set PROXYCURL_API_KEY."""
    if not PROXYCURL_API_KEY:
        return []
    headers = {"Authorization": f"Bearer {PROXYCURL_API_KEY}"}
    slug = _slug_from_company_url(company_url)
    print(f"Extracting profiles for: {company_name} (Proxycurl)...")
    r = requests.get(
        "https://nubela.co/proxycurl/api/linkedin/company/employees",
        params={"url": company_url, "page_size": min(limit, 10), "employment_status": "current"},
        headers=headers,
        timeout=60,
    )
    r.raise_for_status()
    employees = r.json().get("employees") or []
    rows = []
    for emp in employees[:limit]:
        profile_url = emp.get("profile_url")
        if not profile_url:
            continue
        pr = requests.get(
            "https://nubela.co/proxycurl/api/v2/linkedin",
            params={"url": profile_url},
            headers=headers,
            timeout=60,
        )
        pr.raise_for_status()
        person = pr.json()
        for edu in person.get("education") or []:
            school = (edu.get("school") or edu.get("school_name") or "").strip()
            if school:
                rows.append({
                    "company": company_name,
                    "name": person.get("full_name"),
                    "profile_url": profile_url,
                    "school": school,
                    "degree": edu.get("degree_name"),
                })
        time.sleep(0.5)
    return rows

In [ ]:
all_rows = []
use_api = bool(PROXYCURL_API_KEY)

for company_name, company_url in COMPANIES.items():
    if use_api:
        rows = fetch_employees_proxycurl(company_name, company_url, MAX_EMPLOYEES_PER_COMPANY)
    else:
        rows = fetch_employees_selenium(company_name, company_url, driver, MAX_EMPLOYEES_PER_COMPANY)
    all_rows.extend(rows)
    print(f"  -> {len(rows)} education rows for {company_name}")

if not all_rows:
    raise RuntimeError(
        "No profile data could be extracted.\n"
        "- Selenium: complete login (cells above) and re-run.\n"
        "- Or set PROXYCURL_API_KEY and re-run (no browser).\n"
        "- LinkedIn may show zero employees on /people/ without Premium."
    )

df = pd.DataFrame(all_rows)
df.to_csv(OUTPUT_DIR / "education_rows.csv", index=False)
display(df.head(20))
print(f"Saved {len(df)} rows -> {OUTPUT_DIR / 'education_rows.csv'}")

In [ ]:
import matplotlib.pyplot as plt

for company_name in COMPANIES:
    sub = df[df["company"] == company_name]
    if sub.empty:
        continue
    top = Counter(sub["school"]).most_common(15)
    schools, counts = zip(*top)
    fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(schools))))
    ax.barh(list(schools)[::-1], list(counts)[::-1])
    ax.set_title(f"Top schools — {company_name}")
    ax.set_xlabel("mentions (employee × school)")
    plt.tight_layout()
    plt.show()

## Fallback: manual CSV

If scraping stays blocked, export a people search from LinkedIn (or paste profile URLs) into `schools_output/manual_profiles.csv` with columns `company,name,profile_url` and use Proxycurl per URL, or fill `school` by hand.